# Инференс: запрос → рекомендации

Пользователь (история прослушиваний + профиль) пишет запрос на русском, в ответ — текст и список треков.
Вся логика в пакете `recsys/`, здесь только запуск.

* `experiments.ipynb` — почему такая выдача: разбор по шагам, сравнение вариантов, LLM
* `evaluate.py` — метрики на валидации
* `examples/candgen.ipynb` — проверка сервиса BM25 руками

## 1. Настройки

* `USE_CANDGEN = True` — BM25 с сервера по HTTP (`configs/candgen.yaml`). Адрес и ключ — `BM25_URL`, `BM25_API_KEY`
  (файл `.env` в корне или секреты проекта DataSphere).
* `DATA = "crs"` — настоящий датасет из `DATA_DIR` (файлы целиком или частями `train-00000-of-00078.parquet`);
  `"synthetic"` — сгенерированные данные, работает без файлов.

In [ ]:
USE_CANDGEN = True
DATA = "synthetic"         # "crs" — настоящий датасет
DATA_DIR = "data/full"     # tracks_meta, train, test_public
N_USERS = 300              # сколько пользователей загрузить (None — все, долго)

# адрес и ключ BM25 — из файла .env или секретов DataSphere; можно и здесь:
# import os; os.environ["BM25_URL"] = "http://<IP>:8000"; os.environ["BM25_API_KEY"] = "..."

In [ ]:
%pip install -q -r requirements.txt

## 2. Каталог и пайплайн

Загружается один раз. Для сервисов печатается, доступны ли они и какая версия индекса.

In [ ]:
from dataclasses import replace

from recsys.config import load_config
from recsys.data import load_data
from recsys.pipeline import Pipeline
from recsys.schemas import Message

files = ["configs/default.yaml"] + (["configs/candgen.yaml"] if USE_CANDGEN else [])
cfg = load_config(files, {"data": {"source": DATA, "crs": {"dir": DATA_DIR, "n_users": N_USERS}}})
data = load_data(cfg)
pipe = Pipeline.from_config(cfg, data.catalog)

for source in pipe.retrievers:
    if getattr(source, "remote", False):
        print(source.name, source.health())

## 3. Запрос

Берём пользователя из датасета: его история и профиль. `TEXT` — свой запрос; `None` — запрос из датасета
(тогда известен трек, который пользователь на самом деле послушал).

In [ ]:
user = data.requests[0]
TEXT = "хочу что-нибудь спокойное под вечер, без рэпа"     # None — запрос из датасета

request = replace(user, dialog=[Message("user", TEXT)]) if TEXT else user
print("запрос:", request.dialog[-1].text)
print("треков в истории:", len(request.history))
print("профиль:", request.user_info[:200], "...")

## 4. Ответ

In [ ]:
response = pipe.run(request, debug=True)
print(response.text)
response.to_frame()

Если запрос из датасета: на каком месте трек, который пользователь послушал.

In [ ]:
if not TEXT:
    target = request.target_ids[0]
    place = response.track_ids.index(target) + 1 if target in response.track_ids else "нет в выдаче"
    print("цель:", request.meta["artist"], "—", request.meta["title"], "| место:", place)

## 5. Почему такая выдача

Как понят запрос (английские теги — именно они уходят в BM25) и сколько кандидатов дал каждый источник.
`relisten` — треки из своей истории, `bm25_*` — сервер BM25, остальные считаются здесь.

In [ ]:
print("понял как:", response.summary.text)
print("теги:", response.summary.include_tags, "| исключить:", response.summary.exclude_tags)
print("кандидатов:", response.debug["n_candidates"])

## 6. Следующая реплика

Диалог продолжается: уже показанное не повторяется, новые пожелания перекрывают старые.

In [ ]:
request2 = request.next_turn(response, "а теперь поэнергичнее")
response2 = pipe.run(request2)
print(response2.text)
response2.to_frame()